### Polygon Critical Infrastructure Data to Hexbin

In [ ]:
import os

os.chdir("")

os.getcwd()

In [3]:
import numpy as np
from shapely.geometry import Point, Polygon
import pandas as pd
import matplotlib.pyplot as plt
import folium
import base64
import geopandas as gpd
import pathlib as Path
import geopandas as gpd
from shapely import union_all
import fiona
#import gdal

In [ ]:
#Read the water polygon data - data we cleaned and put in a cleaned data folder
reservoirs = gpd.read_file(r'Water.shp')

#Read the hex buff
hex_buff = gpd.read_file(r'.gpkg')

#ensure same CRS
reservoirs = reservoirs.to_crs(hex_buff.crs)

#Find out how they overlay
joined = gpd.sjoin(
    reservoirs,
    hex_buff,
    how="inner",
    predicate="intersects"   # reservoir might cross hex boundary
)

#invent presence field
joined["reservoir_present"] = 1
#Aggregate to hex bins
hex_summary = (
    joined.groupby("h3_ID")["reservoir_present"]
    .max()   # max works for 0/1 → gives 1 if any reservoir exists
    .reset_index()
)
#Join back to grid
hex_with_reservoirs = hex_buff.merge(
    hex_summary,
    on="h3_ID",
    how="left"
)
#Fill 0's 
hex_with_reservoirs["reservoir_present"] = (
    hex_with_reservoirs["reservoir_present"]
    .fillna(0)
    .astype(int)
)



#Save to file
#Save somewhere you know Hex Edits are - 
hex_with_reservoirs.to_file(r'',
    driver="GPKG")